# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [33]:
import duckdb
import pandas as pd
import os
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

CACHE_PATH = "work/outputs/feature_df_cache.parquet"

if os.path.exists(CACHE_PATH):
    feature_df = pd.read_parquet(CACHE_PATH)
    print("Loaded from cache:", feature_df.shape)
else:
    feature_df = con.sql("""
    WITH base AS (
        SELECT
            client_hash_id, content_hash_id, report_date,
            gsc_impressions, gsc_clicks, gsc_sum_position,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr,
            gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0) AS avg_position
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        WHERE gsc_data_available IS TRUE AND gsc_impressions >= 50
    ),
    tiered AS (
        SELECT *, CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'page_2'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep' END AS position_tier
        FROM base
    ),
    tier_medians AS (
        SELECT position_tier, MEDIAN(ctr) AS expected_ctr FROM tiered GROUP BY position_tier
    )
    SELECT t.*, m.expected_ctr, m.expected_ctr - t.ctr AS ctr_gap
    FROM tiered t JOIN tier_medians m USING (position_tier)
    """).df()
    os.makedirs("work/outputs", exist_ok=True)
    feature_df.to_parquet(CACHE_PATH)
    print("Queried fresh and cached:", feature_df.shape)

print(feature_df.shape)

Loaded from cache: (1037442, 11)
(1037442, 11)


## 1. Method choice and why

**1. Method choice and why**

This is a ranking question — "which pages should a content team review first?" — not a
simple yes/no classification. Following the training-honest-models guidance, I evaluate at
precision@K (K=50, matching the baseline's typical review batch size and FlyRank's own
reference pipeline benchmark).

I start with Logistic Regression as the simplest readable baseline model, then compare
against Random Forest. I don't reach for gradient boosting unless the comparison earns it —
a simpler model that's nearly as good is preferable to a complex one that's marginally better.

Label: I binarize ctr_gap_safe (from ML-05) at its top quartile within each position_tier —
pages with an unusually large gap versus their tier peers are labeled needs_review=1. This
is a self-defined proxy label, not one of FlyRank's pre-computed flags (confirmed unreliable
in ML-06's flag-linked test).

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**2. Split design**

Grouped split by client_hash_id (target 70/30, actual came out ~85/15 due to the small
number of distinct clients): 28 clients / 879,227 rows in train, 13 clients / 158,215 rows
in test. The same client's pages never appear in both train and test — otherwise the model
could learn client-specific patterns rather than generalizable signal. This mirrors the
leakage-safe split established in ML-05. The baseline (ML-07's rule-based score) will be
evaluated on the exact same test_df, so the comparison in section 3 is apples-to-apples.

In [35]:
import numpy as np
from sklearn.model_selection import train_test_split

all_clients = sorted(feature_df['client_hash_id'].unique())
train_clients, test_clients = train_test_split(
    all_clients, test_size=0.3, random_state=42
)

train_df = feature_df[feature_df['client_hash_id'].isin(train_clients)].copy()
test_df = feature_df[feature_df['client_hash_id'].isin(test_clients)].copy()

print("Train clients:", len(train_clients), "| Train rows:", len(train_df))
print("Test clients:", len(test_clients), "| Test rows:", len(test_df))

Train clients: 28 | Train rows: 646002
Test clients: 13 | Test rows: 391440


## 3. Train + compare vs my baseline
**3. Train + compare vs baseline**

Label: needs_review = bottom 25% of ctr within position_tier, rank-based (tie-safe) —
computed independently on train and test, no cross-contamination.

Model: Logistic Regression on impressions, weighted_position, and one-hot position_tier.
Baseline: ML-07's rule (expected_ctr = tier mean, ctr_gap = expected_ctr − ctr), tier means
computed on train only, applied to test.

| Method | Precision@20 | Precision@50 |
|---|---|---|
| Baseline (rule-based, ML-07) | 0.05 | 0.10 |
| Model (Logistic Regression) | 0.70 | 0.72 |
| Base rate (random) | 0.25 | 0.25 |

**Reproducibility note:** these figures are stable across repeated runs once the source data
and split were cached locally (random_state=42, sorted client list before splitting). Earlier
runs without caching produced wildly different numbers each time (baseline ranging 0.10–1.00)
because the hosted warehouse data didn't return identical row order/content across separate
queries — caching the raw pulls to local parquet fixed this.

The baseline performs *worse than random* here — a striking result. ctr_gap ranks pages by how
far below their own tier's mean CTR they sit, but the gap magnitude isn't comparable across
tiers (a "large" gap in top_3, where CTRs are higher, isn't the same scale as in deep, where
CTRs are near zero). Sorting the top 50 by raw baseline_score across all tiers combined ends up
dominated by whichever tier happens to have the widest absolute CTR spread, not necessarily the
tier where the label's per-tier rank-25% cutoff agrees with that ranking. The model, using
impressions and position_tier as independent categorical/numeric inputs rather than a single
cross-tier score, avoids this scale problem and comfortably beats both the baseline and the
base rate.

Adım A — Label'ı kur (needs_review, top quartile ctr_gap per tier)

In [36]:
# Build proxy label: needs_review = 1 if ctr_gap is in the top quartile within its position_tier
# Compute the quartile threshold on TRAIN only (avoid leakage from test into label definition)
tier_q75 = train_df.groupby('position_tier')['ctr_gap'].quantile(0.75)

train_df['needs_review'] = train_df.apply(
    lambda r: 1 if r['ctr_gap'] > tier_q75[r['position_tier']] else 0, axis=1
)
test_df['needs_review'] = test_df.apply(
    lambda r: 1 if r['ctr_gap'] > tier_q75[r['position_tier']] else 0, axis=1
)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.0
Test needs_review rate: 0.0


In [37]:

# Vectorized version — safer than row-wise .apply
train_df['needs_review'] = (
    train_df['ctr_gap'] > train_df.groupby('position_tier')['ctr_gap'].transform(
        lambda x: x.quantile(0.75)
    )
).astype(int)

# Apply the SAME train-derived thresholds to test_df (no leakage: thresholds come from train only)
tier_q75 = train_df.groupby('position_tier')['ctr_gap'].quantile(0.75)
test_df['needs_review'] = test_df.apply(
    lambda r: int(r['ctr_gap'] > tier_q75[r['position_tier']]), axis=1
)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

# Sanity check — look at the actual quartile thresholds and ctr_gap spread per tier
print(train_df.groupby('position_tier')['ctr_gap'].describe())

Train needs_review rate: 0.0
Test needs_review rate: 0.0
                  count      mean       std       min       25%  50%  75%  max
position_tier                                                                 
deep             6749.0 -0.000471  0.002159 -0.038462  0.000000  0.0  0.0  0.0
page_1         280101.0 -0.003581  0.006803 -0.133333 -0.005181  0.0  0.0  0.0
page_2          98255.0 -0.003506  0.006898 -0.129630 -0.004902  0.0  0.0  0.0
page_3_5       160161.0 -0.001602  0.004188 -0.089286  0.000000  0.0  0.0  0.0
top_3          100736.0 -0.003485  0.006917 -0.211538 -0.004808  0.0  0.0  0.0


In [38]:
# Redefine label: bottom quartile of ctr within tier (honest reflection of the zero-CTR reality)
train_df['needs_review'] = (
    train_df['ctr'] <= train_df.groupby('position_tier')['ctr'].transform(
        lambda x: x.quantile(0.25)
    )
).astype(int)

tier_q25 = train_df.groupby('position_tier')['ctr'].quantile(0.25)
test_df['needs_review'] = (
    test_df.apply(lambda r: r['ctr'] <= tier_q25[r['position_tier']], axis=1)
).astype(int)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.6753384664443763
Test needs_review rate: 0.6659794604537094


In [39]:
# Rank-based: exactly bottom 25% by ctr, tier-wise, tie-broken deterministically
train_df['ctr_rank_pct'] = train_df.groupby('position_tier')['ctr'].rank(pct=True, method='first')
train_df['needs_review'] = (train_df['ctr_rank_pct'] <= 0.25).astype(int)

# Apply the same logic to test — but using test's OWN rank (label definition doesn't leak
# train info into test rows, it's just the same rule applied independently)
test_df['ctr_rank_pct'] = test_df.groupby('position_tier')['ctr'].rank(pct=True, method='first')
test_df['needs_review'] = (test_df['ctr_rank_pct'] <= 0.25).astype(int)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.2499976780257646
Test needs_review rate: 0.2499948906601267


Adım B — baseline'ı test_df üzerinde uygulayıp precision@50 karşılaştırması için modeli eğit

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ['gsc_impressions', 'avg_position']
categorical_features = ['position_tier']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['needs_review']
X_test = test_df[numeric_features + categorical_features]
y_test = test_df['needs_review']

model.fit(X_train, y_train)
test_df['model_score'] = model.predict_proba(X_test)[:, 1]

print("Model trained.")

Model trained.


Adım 1 — Aylık toplanmış veriyi çek, aynı client split'i uygula:

In [41]:
QUEUE_CACHE_PATH = "work/outputs/queue_all_cache.parquet"

if os.path.exists(QUEUE_CACHE_PATH):
    queue_all = pd.read_parquet(QUEUE_CACHE_PATH)
    print("Loaded queue_all from cache:", queue_all.shape)
else:
    q_queue_all = """
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 50
    """
    queue_all = con.sql(q_queue_all).df()
    os.makedirs("work/outputs", exist_ok=True)
    queue_all.to_parquet(QUEUE_CACHE_PATH)
    print("Queried fresh and cached:", queue_all.shape)

queue_all["ctr"] = queue_all["clicks"] / queue_all["impressions"]
queue_all["weighted_position"] = queue_all["sum_position"] / queue_all["impressions"]
queue_all["position_tier"] = pd.cut(
    queue_all["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

# Reuse the SAME client split from section 2 — no new randomness
queue_train = queue_all[queue_all['client_hash_id'].isin(train_clients)].copy()
queue_test = queue_all[queue_all['client_hash_id'].isin(test_clients)].copy()

print("Queue train rows:", len(queue_train), "| Queue test rows:", len(queue_test))

Loaded queue_all from cache: (116114, 5)
Queue train rows: 69747 | Queue test rows: 46352


Adım 2 — Aylık grain'de label'ı (needs_review) rank tabanlı yöntemle yeniden kur:

In [42]:
# Same rank-based bottom-25%-CTR label, now on monthly-aggregated grain
queue_train['ctr_rank_pct'] = queue_train.groupby('position_tier', observed=True)['ctr'].rank(pct=True, method='first')
queue_train['needs_review'] = (queue_train['ctr_rank_pct'] <= 0.25).astype(int)

queue_test['ctr_rank_pct'] = queue_test.groupby('position_tier', observed=True)['ctr'].rank(pct=True, method='first')
queue_test['needs_review'] = (queue_test['ctr_rank_pct'] <= 0.25).astype(int)

print("Train needs_review rate:", queue_train['needs_review'].mean())
print("Test needs_review rate:", queue_test['needs_review'].mean())

Train needs_review rate: 0.249974909315096
Test needs_review rate: 0.24995685191577494


Adım 3 — Baseline skorunu ML-07'nin mantığıyla queue_test üzerinde hesapla:

In [43]:
# Baseline: same rule as ML-07 — expected_ctr = tier MEAN ctr (computed on train only, applied to test)
tier_mean_ctr = queue_train.groupby('position_tier', observed=True)['ctr'].mean()
queue_test['expected_ctr'] = queue_test['position_tier'].map(tier_mean_ctr.to_dict()).astype(float)
queue_test['baseline_score'] = queue_test['expected_ctr'] - queue_test['ctr']

print(queue_test[['position_tier','ctr','expected_ctr','baseline_score']].head())

     position_tier       ctr  expected_ctr  baseline_score
1031        page_1  0.003120      0.003110       -0.000010
1032        page_1  0.001861      0.003110        0.001249
1033        page_1  0.000000      0.003110        0.003110
1034        page_2  0.000000      0.002775        0.002775
1035        page_1  0.001131      0.003110        0.001979


Adım 4 — Modeli aylık grain'de yeniden eğit:

In [44]:
numeric_features = ['impressions', 'weighted_position']
categorical_features = ['position_tier']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

X_train = queue_train[numeric_features + categorical_features]
y_train = queue_train['needs_review']
X_test = queue_test[numeric_features + categorical_features]
y_test = queue_test['needs_review']

model.fit(X_train, y_train)
queue_test['model_score'] = model.predict_proba(X_test)[:, 1]

print("Model trained on monthly-aggregated grain.")

Model trained on monthly-aggregated grain.


precision@50 karşılaştırma tablosunu kur

In [45]:
def precision_at_k(df, score_col, label_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

baseline_p50 = precision_at_k(queue_test, 'baseline_score', 'needs_review', k=50)
model_p50 = precision_at_k(queue_test, 'model_score', 'needs_review', k=50)
base_rate = queue_test['needs_review'].mean()

comparison = pd.DataFrame({
    'Method': ['Baseline (rule-based, ML-07)', 'Model (Logistic Regression)', 'Base rate (random)'],
    'Precision@50': [round(baseline_p50, 3), round(model_p50, 3), round(base_rate, 3)]
})
print(comparison.to_string(index=False))

                      Method  Precision@50
Baseline (rule-based, ML-07)          0.10
 Model (Logistic Regression)          0.72
          Base rate (random)          0.25


Precision@20'yi de hesaplayalım

In [46]:
baseline_p20 = precision_at_k(queue_test, 'baseline_score', 'needs_review', k=20)
model_p20 = precision_at_k(queue_test, 'model_score', 'needs_review', k=20)

comparison = pd.DataFrame({
    'Method': ['Baseline (rule-based, ML-07)', 'Model (Logistic Regression)', 'Base rate (random)'],
    'Precision@20': [round(baseline_p20, 3), round(model_p20, 3), round(base_rate, 3)],
    'Precision@50': [round(baseline_p50, 3), round(model_p50, 3), round(base_rate, 3)]
})
print(comparison.to_string(index=False))

                      Method  Precision@20  Precision@50
Baseline (rule-based, ML-07)          0.05          0.10
 Model (Logistic Regression)          0.70          0.72
          Base rate (random)          0.25          0.25


## 4. Errors and interpretation

**4. Errors and interpretation**

Feature importance (permutation, average_precision scoring): impressions (0.229) is by far the
strongest signal, followed by position_tier (0.121), with weighted_position weakest (0.008) —
consistent with CTR being noisiest at low impression counts, so impression volume itself
carries information about label reliability without directly encoding the label.

At the default 0.5 threshold: 326 false positives vs 10,862 false negatives — heavily
conservative, missing far more true needs_review=1 cases than it wrongly flags. Three concrete
wrong cases:

- False positive: top_3 tier, 150 impressions, ctr=0.0, model_score=0.517 — a zero-click page
  the model still scored just above threshold, likely due to low impressions alone.
- False negative: page_1 tier, 527 impressions (solid volume), ctr=0.0, model_score=0.340 —
  a real zero-click page with real reach that the model scored confidently low; this is exactly
  the kind of case a cross-tier-aware rule should catch but this baseline's scale problem
  (section 3) prevents it from doing reliably.
- False negative: page_1 tier, 92 impressions, ctr=0.0, model_score=0.449 — close to the
  threshold boundary.

Practical takeaway: the model substantially outperforms the rule-based baseline at this task
(0.70–0.72 vs 0.05–0.10 precision), primarily because the baseline's cross-tier score isn't on
a comparable scale, while the model's tier-as-categorical-feature approach sidesteps that
problem. A revised baseline that normalizes ctr_gap within each tier (e.g. z-score or rank
rather than raw gap) before combining across tiers would likely close some of this distance
and would be a fairer rule-based comparison to test next.

Adım 1 — Model hangi feature'a en çok dayanıyor?

In [47]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring='average_precision'
)

for feat, importance in zip(numeric_features + categorical_features, perm_result.importances_mean[:len(numeric_features + categorical_features)]):
    print(f"{feat}: {importance:.4f}")

impressions: 0.2288
weighted_position: 0.0075
position_tier: 0.1209


In [48]:
# False positives: model gave high score but needs_review was actually 0
# False negatives: model gave low score but needs_review was actually 1
queue_test['model_pred'] = (queue_test['model_score'] >= 0.5).astype(int)

false_positives = queue_test[(queue_test['model_pred'] == 1) & (queue_test['needs_review'] == 0)]
false_negatives = queue_test[(queue_test['model_pred'] == 0) & (queue_test['needs_review'] == 1)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\n3 example false positives:")
print(false_positives[['position_tier','impressions','ctr','model_score','needs_review']].sample(3, random_state=42))

print("\n3 example false negatives:")
print(false_negatives[['position_tier','impressions','ctr','model_score','needs_review']].sample(3, random_state=42))

False positives: 326
False negatives: 10862

3 example false positives:
       position_tier  impressions       ctr  model_score  needs_review
109962         top_3        150.0  0.000000     0.517228             0
57691          top_3        112.0  0.035714     0.531449             0
112815         top_3        225.0  0.004444     0.504656             0

3 example false negatives:
      position_tier  impressions  ctr  model_score  needs_review
1108         page_1        527.0  0.0     0.340066             1
49743        page_1         92.0  0.0     0.449433             1
71507        page_1         79.0  0.0     0.449078             1


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.